# SQL Debugging & Code Review — PostgreSQL

**Purpose:** This guide is a quick-reference for reading, classifying, and fixing common SQL errors in PostgreSQL — structured for a hands-on technical interview where you'll debug queries and review code under time pressure. Every error pattern includes the exact message you'll see, what caused it, the fix, and how to verify it worked.

---

### Table of Contents

**[1. How to Read a PostgreSQL Error](#1-how-to-read-a-postgresql-error)** — Anatomy of an error message

**[2. Error Classification Decision Tree](#2-error-classification-decision-tree)** — Route from symptom to category in seconds

**[3. Error Pattern Library](#3-error-pattern-library)**
- [A. Syntax Errors](#pattern-a-syntax-errors) — Missing commas, misplaced keywords, bad aliases
- [B. Column & Table Reference Errors](#pattern-b-reference-errors) — Ambiguous columns, missing tables, typos
- [C. Type Mismatch & Casting Errors](#pattern-c-type-errors) — Implicit casts, string vs. integer, date formats
- [D. Aggregation & GROUP BY Errors](#pattern-d-aggregation-errors) — Missing GROUP BY columns, WHERE vs HAVING
- [E. NULL Traps](#pattern-e-null-traps) — Invisible NULLs breaking COUNT, SUM, comparisons
- [F. JOIN & Row Count Errors](#pattern-f-join-errors) — Fan-out, missing ON clause, Cartesian products
- [G. Window Function Errors](#pattern-g-window-errors) — Wrong frame, missing OVER, nested aggregates
- [H. Subquery Errors](#pattern-h-subquery-errors) — Multi-row returns, correlated misuse, missing alias
- [I. Division by Zero & Arithmetic Errors](#pattern-i-arithmetic-errors) — NULLIF, COALESCE, safe division

**[4. Code Review Checklist](#4-code-review-checklist)** — What to look for when reviewing someone else's SQL

**[5. Debugging Workflow Under Pressure](#5-debugging-workflow)** — Systematic 5-step process for interview debugging

**[6. Common Interview Traps](#6-interview-traps)** — Errors interviewers plant deliberately

<hr style="border: 3px solid black;">

<a id='1-how-to-read-a-postgresql-error'></a>

## 1. How to Read a PostgreSQL Error

PostgreSQL error messages follow a consistent structure. Knowing where to look saves you from guessing.

```
ERROR:  column "department_name" must appear in the GROUP BY clause
        or be used in an aggregate function
LINE 3:     department_name,
            ^
DETAIL:  ...
HINT:    Add department_name to the GROUP BY clause.
```

---

**Anatomy of the Error**

| Part | What It Tells You | Where to Look |
|---|---|---|
| `ERROR:` | The main error type and description | Read this first — it's the classification |
| `LINE N:` | The exact line number where the error was detected | Jump to this line in your query |
| `^` (caret) | Points to the exact character position | The problem is at or just before the caret |
| `DETAIL:` | Additional context about the cause | Not always present; read when available |
| `HINT:` | PostgreSQL's suggested fix | Often exactly right — don't ignore it |

---

**The 3-Second Read**

When you see an error in an interview, read it in this order:

1. **ERROR line** — What category of problem is this? (syntax? type? reference?)
2. **LINE number** — Where in the query?
3. **HINT** — Does PostgreSQL already know the fix?

> **Interview tip:** Read the error message out loud to the interviewer. It shows you're systematic, not guessing. Say: "The error says column X must appear in the GROUP BY clause, on line 3 — so I need to either add it to GROUP BY or wrap it in an aggregate.".

<hr style="border: 3px solid black;">

<a id='2-error-classification-decision-tree'></a>

## 2. Error Classification Decision Tree

Use this when you see an error and need to classify it fast:

<div class="fc">
  <div class="fc-node fc-start">READ THE ERROR MESSAGE<br/>What does the ERROR line say?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">"syntax error at or near"?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern A — Syntax</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"column does not exist" or "relation does not exist" or "ambiguous"?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern B — Reference</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"cannot cast" or "operator does not exist"?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern C — Type Mismatch</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"must appear in GROUP BY" or "not in aggregate"?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern D — Aggregation</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Query runs but results are wrong?<br/>(unexpected NULLs, wrong counts, missing rows)</div>
  <div class="fc-node fc-warn">CHECK BOTH → <strong>Pattern E — NULL Traps</strong><br/><strong>Pattern F — JOIN Errors</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"window function in WHERE" or wrong ranking results?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern G — Window Errors</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"subquery returns more than one row" or correlated subquery issues?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern H — Subquery Errors</strong></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"division by zero"?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern I — Arithmetic</strong></div>
</div>

> **Key insight:** Errors fall into two buckets — **hard errors** (the query won't run) and **silent errors** (the query runs but returns wrong results). Silent errors are harder and more common in interviews.

<hr style="border: 3px solid black;">

<a id='3-error-pattern-library'></a>

## 3. Error Pattern Library

Each pattern follows the same structure: **Error message → What caused it → The fix → Verify it worked**.

<a id='pattern-a-syntax-errors'></a>

### A. Syntax Errors

**Error messages:** `syntax error at or near "..."`, `unexpected token`

These are the most common errors — a missing comma, misplaced keyword, or unclosed parenthesis.

---

**A1. Missing comma between SELECT columns**

```sql
-- BROKEN
SELECT
    employee_id
    employee_name    -- ← missing comma after employee_id
FROM Employees;
```

```
ERROR:  syntax error at or near "employee_name"
LINE 3:     employee_name
            ^
```

**Fix:** Add the comma. **Verify:** Query runs and returns expected columns.

```sql
SELECT
    employee_id,
    employee_name
FROM Employees;
```

---

**A2. Using a reserved keyword as an alias**

```sql
-- BROKEN
SELECT
    COUNT(*) AS order,    -- ← "order" is a reserved keyword
    customer_id
FROM Orders
GROUP BY customer_id;
```

```
ERROR:  syntax error at or near "order"
```

**Fix:** Quote the alias or choose a different name.

```sql
SELECT
    COUNT(*) AS "order",       -- quoted (works but awkward)
    COUNT(*) AS order_count    -- better: just rename it
FROM Orders
GROUP BY customer_id;
```

---

**A3. Misplaced or missing keyword**

```sql
-- BROKEN: WHERE before JOIN
SELECT e.name, d.department_name
FROM Employees e
WHERE e.salary > 50000
INNER JOIN Departments d ON e.dept_id = d.dept_id;
```

```
ERROR:  syntax error at or near "INNER"
```

**Fix:** SQL clause order is `FROM → JOIN → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`.

```sql
SELECT e.name, d.department_name
FROM Employees e
INNER JOIN Departments d ON e.dept_id = d.dept_id
WHERE e.salary > 50000;
```

---

**A4. Unclosed parenthesis**

```sql
-- BROKEN
SELECT *
FROM Orders
WHERE status IN ('active', 'pending';   -- ← missing closing )
```

```
ERROR:  syntax error at or near ";"
```

**Fix:** Count your parentheses. The caret points to the `;` because PostgreSQL was still expecting `)`.

---

**Quick checklist for syntax errors:**

| Symptom | Likely Cause |
|---|---|
| Error points to a column name | Missing comma on the line above |
| Error points to a keyword (JOIN, GROUP, ORDER) | Wrong clause order |
| Error points to `;` | Unclosed parenthesis or quote |
| Error mentions a common word (order, table, user, group) | Reserved keyword used as alias |

<hr style="border: 2px solid black;">

<a id='pattern-b-reference-errors'></a>

### B. Column & Table Reference Errors

**Error messages:** `column "X" does not exist`, `relation "X" does not exist`, `column reference "X" is ambiguous`

---

**B1. Ambiguous column in a JOIN**

```sql
-- BROKEN
SELECT customer_id, name, order_date    -- ← customer_id exists in both tables
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id;
```

```
ERROR:  column reference "customer_id" is ambiguous
```

**Fix:** Prefix every column with its table alias.

```sql
SELECT c.customer_id, c.name, o.order_date
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id;
```

---

**B2. Using a SELECT alias in WHERE**

```sql
-- BROKEN
SELECT
    salary * 12 AS annual_salary
FROM Employees
WHERE annual_salary > 100000;    -- ← alias not available in WHERE
```

```
ERROR:  column "annual_salary" does not exist
```

**Fix:** PostgreSQL evaluates `WHERE` before `SELECT`, so the alias doesn't exist yet. Repeat the expression or use a CTE.

```sql
-- Option 1: Repeat the expression
SELECT salary * 12 AS annual_salary
FROM Employees
WHERE salary * 12 > 100000;

-- Option 2: CTE (cleaner for complex expressions)
WITH emp AS (
    SELECT *, salary * 12 AS annual_salary
    FROM Employees
)
SELECT * FROM emp WHERE annual_salary > 100000;
```

---

**B3. Alias not available in GROUP BY (in some contexts)**

```sql
-- PostgreSQL ALLOWS aliases in GROUP BY (unlike some other databases)
-- But this is dialect-specific — in MySQL it works, in SQL Server it doesn't
SELECT EXTRACT(MONTH FROM order_date) AS order_month, COUNT(*)
FROM Orders
GROUP BY order_month;    -- ← This works in PostgreSQL
```

> **Interview tip:** Mention this is PostgreSQL-specific. In a cross-dialect environment, use the full expression in GROUP BY to be safe.

---

**B4. Referencing a column from a table not in FROM**

```sql
-- BROKEN: forgot to join the Departments table
SELECT e.name, d.department_name
FROM Employees e
WHERE e.salary > 50000;
```

```
ERROR:  missing FROM-clause entry for table "d"
```

**Fix:** Either add the JOIN or remove the reference.

---

**Reference error decision tree:**

<div class="fc">
  <div class="fc-node fc-start">Column / table error?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-four">
    <div class="fc-branch">
      <div class="fc-label fc-tag">"is ambiguous"</div>
      <div class="fc-node fc-good">Add table alias prefix</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">"does not exist" in WHERE</div>
      <div class="fc-node fc-good">Alias from SELECT?<br/>Repeat expression or CTE</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">"does not exist" in SELECT</div>
      <div class="fc-node fc-good">Typo? Check column name</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">"missing FROM-clause"</div>
      <div class="fc-node fc-good">Forgot to JOIN that table</div>
    </div>
  </div>
</div>

<hr style="border: 2px solid black;">

<a id='pattern-c-type-errors'></a>

### C. Type Mismatch & Casting Errors

**Error messages:** `operator does not exist: varchar = integer`, `cannot cast type`, `invalid input syntax for type`

---

**C1. Comparing a string column to an integer**

```sql
-- BROKEN: zip_code is VARCHAR but compared to integer
SELECT * FROM Customers WHERE zip_code = 90210;
```

```
ERROR:  operator does not exist: character varying = integer
HINT:  No operator matches the given name and argument types.
       You might need to add explicit type casts.
```

**Fix:** Match the type — use a string literal.

```sql
SELECT * FROM Customers WHERE zip_code = '90210';
```

---

**C2. Aggregating a non-numeric column**

```sql
-- BROKEN
SELECT SUM(status) FROM Orders;    -- status is VARCHAR
```

```
ERROR:  function sum(character varying) does not exist
```

**Fix:** You probably want `COUNT` not `SUM`, or you need `CASE` to convert to numeric first.

```sql
-- Count occurrences
SELECT COUNT(*) FROM Orders WHERE status = 'active';

-- Or sum a derived numeric value
SELECT SUM(CASE WHEN status = 'active' THEN 1 ELSE 0 END) FROM Orders;
```

---

**C3. Date format mismatch**

```sql
-- BROKEN: ambiguous date format
SELECT * FROM Orders WHERE order_date = '03/04/2024';
-- Is this March 4 or April 3? Depends on locale settings.
```

**Fix:** Always use ISO 8601 format (`YYYY-MM-DD`).

```sql
SELECT * FROM Orders WHERE order_date = '2024-03-04';
```

---

**C4. Integer division returning 0 instead of decimal**

```sql
-- BROKEN (silent error — query runs but result is wrong)
SELECT completed_tasks / total_tasks AS completion_rate
FROM Projects;
-- If both are integers: 3 / 5 = 0, not 0.6
```

**Fix:** Cast one operand to a decimal type.

```sql
SELECT completed_tasks::DECIMAL / total_tasks AS completion_rate
FROM Projects;
-- Or: CAST(completed_tasks AS DECIMAL) / total_tasks
```

---

**Type error quick reference:**

| Error Pattern | Usual Cause | Fix |
|---|---|---|
| `operator does not exist: varchar = integer` | Comparing string to number | Add quotes or cast |
| `function sum(varchar)` | Aggregating a text column | Use COUNT or CASE |
| `invalid input syntax for type date` | Wrong date format | Use `'YYYY-MM-DD'` |
| Division returns 0 | Integer division | Cast to `DECIMAL` or `NUMERIC` |

<hr style="border: 2px solid black;">

<a id='pattern-d-aggregation-errors'></a>

### D. Aggregation & GROUP BY Errors

**Error messages:** `must appear in the GROUP BY clause or be used in an aggregate function`, `aggregate functions are not allowed in WHERE`

These are the most common interview errors — they come up in nearly every GROUP BY problem.

---

**D1. SELECT column not in GROUP BY**

```sql
-- BROKEN
SELECT department_id, employee_name, COUNT(*)
FROM Employees
GROUP BY department_id;
-- ← employee_name is not grouped or aggregated
```

```
ERROR:  column "employees.employee_name" must appear in the GROUP BY
        clause or be used in an aggregate function
```

**Fix:** Either add it to GROUP BY or wrap it in an aggregate.

```sql
-- Option 1: Add to GROUP BY (if you want per-employee counts)
SELECT department_id, employee_name, COUNT(*)
FROM Employees
GROUP BY department_id, employee_name;

-- Option 2: Remove it (if you want per-department counts)
SELECT department_id, COUNT(*)
FROM Employees
GROUP BY department_id;
```

---

**D2. Using aggregate in WHERE instead of HAVING**

```sql
-- BROKEN
SELECT department_id, COUNT(*) AS emp_count
FROM Employees
WHERE COUNT(*) > 5    -- ← aggregate not allowed in WHERE
GROUP BY department_id;
```

```
ERROR:  aggregate functions are not allowed in WHERE
```

**Fix:** `WHERE` filters rows *before* grouping. `HAVING` filters groups *after* aggregation.

```sql
SELECT department_id, COUNT(*) AS emp_count
FROM Employees
GROUP BY department_id
HAVING COUNT(*) > 5;
```

---

**D3. Using a SELECT alias in HAVING**

```sql
-- BROKEN in standard SQL (but PostgreSQL actually allows this)
SELECT department_id, COUNT(*) AS emp_count
FROM Employees
GROUP BY department_id
HAVING emp_count > 5;    -- ← Works in PostgreSQL, fails in SQL Server
```

> **Interview tip:** For maximum portability, use the full expression in HAVING: `HAVING COUNT(*) > 5`.

---

**D4. ORDER BY a non-selected, non-grouped column**

```sql
-- BROKEN
SELECT department_id, COUNT(*) AS emp_count
FROM Employees
GROUP BY department_id
ORDER BY employee_name;    -- ← not in GROUP BY or aggregate
```

```
ERROR:  column "employees.employee_name" must appear in the GROUP BY
        clause or be used in an aggregate function
```

**Fix:** ORDER BY a column that's in SELECT or wrap it in an aggregate (e.g., `ORDER BY MAX(employee_name)`).

---

**The WHERE vs. HAVING Rule**

<div class="fc">
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">WHERE</div>
      <div class="fc-node fc-action">Filters <strong>ROWS</strong><br/>Runs BEFORE GROUP BY<br/>Cannot use aggregates<br/>e.g., <code>salary > 50000</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">HAVING</div>
      <div class="fc-node fc-action">Filters <strong>GROUPS</strong><br/>Runs AFTER GROUP BY<br/>Can use aggregates<br/>e.g., <code>COUNT(*) > 5</code></div>
    </div>
  </div>
</div>

<hr style="border: 2px solid black;">

<a id='pattern-e-null-traps'></a>

### E. NULL Traps (Silent Errors)

**No error message** — the query runs but results are wrong. These are the most dangerous interview errors because you won't see an error; you have to *know* to check.

---

**E1. COUNT(*) vs COUNT(column)**

```sql
-- These return DIFFERENT numbers when NULLs exist
SELECT
    COUNT(*) AS total_rows,           -- counts ALL rows (5)
    COUNT(email) AS rows_with_email   -- skips NULLs (3)
FROM Customers;
```

| customer_id | email |
|---|---|
| 1 | a@test.com |
| 2 | NULL |
| 3 | b@test.com |
| 4 | NULL |
| 5 | c@test.com |

`COUNT(*) = 5`, `COUNT(email) = 3`. If you use `COUNT(email)` when you meant "total customers," your result is silently wrong.

**Fix:** Use `COUNT(*)` for total rows. Use `COUNT(column)` only when you intentionally want to exclude NULLs.

---

**E2. NULL in NOT IN**

```sql
-- BROKEN: Returns 0 rows if any manager_id is NULL
SELECT * FROM Employees
WHERE employee_id NOT IN (SELECT manager_id FROM Employees);
```

**Why:** If `manager_id` has even one NULL, `NOT IN` evaluates to UNKNOWN for every row, returning nothing.

**Fix:** Use `NOT EXISTS` or filter NULLs.

```sql
-- Safe: NOT EXISTS handles NULLs correctly
SELECT * FROM Employees e
WHERE NOT EXISTS (
    SELECT 1 FROM Employees m WHERE m.manager_id = e.employee_id
);

-- Or: Filter NULLs from the subquery
SELECT * FROM Employees
WHERE employee_id NOT IN (
    SELECT manager_id FROM Employees WHERE manager_id IS NOT NULL
);
```

---

**E3. NULL in arithmetic — SUM, AVG, and comparisons**

```sql
-- NULL + 100 = NULL (not 100)
-- NULL > 5 = NULL (not TRUE or FALSE)
-- AVG ignores NULLs: AVG(10, NULL, 20) = 15, not 10
```

**Fix:** Use `COALESCE` to set a default.

```sql
SELECT COALESCE(bonus, 0) + salary AS total_comp
FROM Employees;
```

---

**E4. NULL in WHERE != comparisons**

```sql
-- BROKEN: Misses rows where status IS NULL
SELECT * FROM Orders WHERE status != 'cancelled';
```

Rows where `status IS NULL` are not returned — because `NULL != 'cancelled'` evaluates to `NULL` (not TRUE).

**Fix:** Explicitly include NULLs.

```sql
SELECT * FROM Orders
WHERE status != 'cancelled' OR status IS NULL;

-- Or use IS DISTINCT FROM (PostgreSQL-specific)
SELECT * FROM Orders
WHERE status IS DISTINCT FROM 'cancelled';
```

---

**NULL trap checklist:**

| Trap | Symptom | Fix |
|---|---|---|
| `COUNT(col)` vs `COUNT(*)` | Row count lower than expected | Use `COUNT(*)` for total rows |
| `NOT IN` with NULLs | Zero rows returned | Use `NOT EXISTS` |
| `!=` missing NULLs | Fewer rows than expected | Add `OR col IS NULL` |
| Arithmetic with NULL | NULL results propagating | Wrap in `COALESCE(col, 0)` |
| `AVG` ignoring NULLs | Average seems too high/low | Decide: treat NULL as 0 or exclude? |

<hr style="border: 2px solid black;">

<a id='pattern-f-join-errors'></a>

### F. JOIN & Row Count Errors (Silent Errors)

**No error message** — the query runs but produces too many or too few rows. The most common cause of wrong aggregations in interviews.

---

**F1. Fan-out — row count inflates after JOIN**

```sql
-- Expected: 100 customers. Got: 350 rows.
SELECT c.customer_id, c.name, o.order_id
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id;
```

**Why:** One customer can have multiple orders. The JOIN duplicates customer rows — one per order.

**Diagnosis:**

```sql
-- Check: how many orders per customer?
SELECT customer_id, COUNT(*) FROM Orders GROUP BY customer_id ORDER BY 2 DESC;
```

**Fix:** Aggregate first, then join.

```sql
WITH order_counts AS (
    SELECT customer_id, COUNT(*) AS order_count
    FROM Orders GROUP BY customer_id
)
SELECT c.customer_id, c.name, oc.order_count
FROM Customers c
JOIN order_counts oc ON c.customer_id = oc.customer_id;
```

---

**F2. Missing ON clause → Cartesian product**

```sql
-- BROKEN: Returns customers × orders (every combination)
SELECT c.name, o.order_date
FROM Customers c, Orders o;    -- ← implicit cross join, missing WHERE
```

**Fix:** Use explicit JOIN with ON clause.

---

**F3. INNER JOIN silently drops rows**

```sql
-- Expected all customers, but 20 are missing
SELECT c.customer_id, o.order_id
FROM Customers c
INNER JOIN Orders o ON c.customer_id = o.customer_id;
```

**Why:** INNER JOIN drops customers who have no orders. If the question says "all customers" or "including those with no orders," use LEFT JOIN.

**Diagnosis:**

```sql
-- How many customers have no orders?
SELECT COUNT(*) FROM Customers c
LEFT JOIN Orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;
```

---

**F4. Duplicate join key causing double-counting**

```sql
-- SUM is 2x expected because the join duplicated rows
SELECT d.department_name, SUM(e.salary)
FROM Departments d
JOIN Employees e ON d.dept_id = e.dept_id
JOIN Projects p ON e.employee_id = p.employee_id    -- ← 1:many
GROUP BY d.department_name;
```

**Why:** If an employee is on 3 projects, their salary gets summed 3 times.

**Fix:** Aggregate in a CTE before the second join, or use `SUM(DISTINCT e.salary)` if salaries are unique.

---

**JOIN error diagnostic flow:**

<div class="fc">
  <div class="fc-node fc-start">Row count wrong after JOIN?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">TOO MANY ROWS</div>
      <div class="fc-node fc-warn">Check for duplicate keys (fan-out)<br/>Missing ON clause (Cartesian product)<br/><strong>Fix:</strong> Aggregate before joining</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">TOO FEW ROWS</div>
      <div class="fc-node fc-warn">INNER JOIN dropping unmatched? → LEFT JOIN<br/>WHERE filtering NULLs? → Move to ON<br/><strong>Fix:</strong> Check with LEFT JOIN + IS NULL</div>
    </div>
  </div>
</div>

<hr style="border: 2px solid black;">

<a id='pattern-g-window-errors'></a>

### G. Window Function Errors

**Error messages:** `window functions are not allowed in WHERE`, `aggregate function calls cannot be nested`

---

**G1. Using a window function in WHERE**

```sql
-- BROKEN
SELECT employee_id, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
FROM Employees
WHERE rn <= 5;    -- ← window function not available in WHERE
```

```
ERROR:  column "rn" does not exist
```

**Fix:** Window functions are computed after WHERE. Wrap in a CTE or subquery.

```sql
WITH ranked AS (
    SELECT employee_id, salary,
           ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
    FROM Employees
)
SELECT * FROM ranked WHERE rn <= 5;
```

---

**G2. Nesting an aggregate inside a window function**

```sql
-- BROKEN
SELECT department_id,
       SUM(COUNT(*)) OVER (ORDER BY department_id)
FROM Employees
GROUP BY department_id;
```

```
ERROR:  aggregate function calls cannot be nested
```

**Fix:** Compute the aggregate in a CTE first, then apply the window.

```sql
WITH dept_counts AS (
    SELECT department_id, COUNT(*) AS emp_count
    FROM Employees GROUP BY department_id
)
SELECT department_id,
       SUM(emp_count) OVER (ORDER BY department_id) AS running_total
FROM dept_counts;
```

---

**G3. Wrong window frame — ROWS vs RANGE (silent error)**

```sql
-- Intended: 7-day rolling average
-- Actual: averages over 7 ROWS, not 7 DAYS
SELECT visited_on,
       AVG(amount) OVER (ORDER BY visited_on ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
FROM Customer;
```

**Why:** If multiple rows share the same date, `ROWS BETWEEN 6 PRECEDING` counts 6 prior rows, not 6 prior days.

**Fix:** Pre-aggregate to one row per date, then use ROWS. See Pattern F (Running & Rolling) in the Single-Table guide.

---

**G4. Missing PARTITION BY — window spans entire table**

```sql
-- Intended: rank within each department
-- Actual: ranks across ALL employees
SELECT employee_id, department_id,
       RANK() OVER (ORDER BY salary DESC) AS dept_rank
FROM Employees;
```

**Fix:** Add `PARTITION BY`.

```sql
RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS dept_rank
```

<hr style="border: 2px solid black;">

<a id='pattern-h-subquery-errors'></a>

### H. Subquery Errors

**Error messages:** `subquery must return only one column`, `subquery used as an expression must return at most one row`, `subquery in FROM must have an alias`

---

**H1. Scalar subquery returns multiple rows**

```sql
-- BROKEN: subquery returns multiple department_ids
SELECT employee_name
FROM Employees
WHERE department_id = (SELECT department_id FROM Departments WHERE location = 'NYC');
```

```
ERROR:  more than one row returned by a subquery used as an expression
```

**Fix:** Use `IN` instead of `=` when the subquery returns multiple rows.

```sql
WHERE department_id IN (SELECT department_id FROM Departments WHERE location = 'NYC');
```

---

**H2. Missing alias on derived table**

```sql
-- BROKEN
SELECT * FROM (SELECT customer_id, COUNT(*) FROM Orders GROUP BY customer_id);
```

```
ERROR:  subquery in FROM must have an alias
```

**Fix:** Always alias subqueries in FROM.

```sql
SELECT * FROM (
    SELECT customer_id, COUNT(*) AS order_count
    FROM Orders GROUP BY customer_id
) AS order_summary;
```

---

**H3. Correlated subquery referencing wrong scope**

```sql
-- BROKEN: e2 alias not visible inside the inner subquery
SELECT e1.name
FROM Employees e1
WHERE e1.salary > (
    SELECT AVG(e2.salary) FROM Employees e2
    WHERE e2.department_id = e3.department_id    -- ← e3 doesn't exist
);
```

**Fix:** Check that all correlated references point to the correct outer alias.

```sql
WHERE e1.salary > (
    SELECT AVG(e2.salary) FROM Employees e2
    WHERE e2.department_id = e1.department_id    -- ← reference outer e1
);
```

<hr style="border: 2px solid black;">

<a id='pattern-i-arithmetic-errors'></a>

### I. Division by Zero & Arithmetic Errors

**Error message:** `division by zero`

---

**I1. Division by zero in a ratio calculation**

```sql
-- BROKEN: total_tasks could be 0
SELECT project_id,
       completed_tasks / total_tasks AS completion_rate
FROM Projects;
```

```
ERROR:  division by zero
```

**Fix:** Use `NULLIF` to convert 0 to NULL (which makes the division return NULL instead of erroring).

```sql
SELECT project_id,
       completed_tasks / NULLIF(total_tasks, 0) AS completion_rate
FROM Projects;
```

---

**I2. Safe division pattern (return 0 instead of NULL)**

```sql
SELECT project_id,
       COALESCE(completed_tasks::DECIMAL / NULLIF(total_tasks, 0), 0) AS completion_rate
FROM Projects;
```

**Pattern breakdown:** `NULLIF(total_tasks, 0)` → returns NULL if zero → division returns NULL → `COALESCE(..., 0)` → converts NULL to 0.

---

**I3. Rounding precision**

```sql
-- PostgreSQL integer division: 3 / 5 = 0 (not 0.6)
SELECT ROUND(3::DECIMAL / 5, 2);    -- = 0.60
```

> **Interview tip:** When computing rates or percentages, always cast to DECIMAL first and use ROUND for clean output.

<hr style="border: 3px solid black;">

<a id='4-code-review-checklist'></a>

## 4. Code Review Checklist

When reviewing someone else's SQL in an interview, scan for these issues in order:

---

**Pass 1 — Structure (5 seconds)**

| Check | What to Look For |
|---|---|
| Clause order | `SELECT → FROM → JOIN → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT` |
| Table aliases | Every table has an alias; aliases are used consistently |
| Indentation | Subqueries and CTEs are properly indented for readability |

**Pass 2 — Logic (30 seconds)**

| Check | What to Look For |
|---|---|
| JOIN type | Should it be LEFT JOIN instead of INNER JOIN? Does the question say "all" or "including"? |
| GROUP BY completeness | Every non-aggregated SELECT column appears in GROUP BY |
| WHERE vs HAVING | Are aggregates used in WHERE? They need to be in HAVING |
| NULL handling | Any `NOT IN` subqueries? Any `!=` that should include `IS NULL`? |
| Alias scope | Is a SELECT alias used in WHERE? (Not valid in standard SQL) |

**Pass 3 — Data Correctness (30 seconds)**

| Check | What to Look For |
|---|---|
| Fan-out | Will the JOIN produce more rows than expected? |
| COUNT(*) vs COUNT(col) | Which one matches the intent? |
| Integer division | Will a ratio return 0 instead of a decimal? |
| Division by zero | Is there a `NULLIF` or `CASE` guard? |
| Date format | ISO 8601? Or ambiguous MM/DD vs DD/MM? |
| DISTINCT necessity | Is DISTINCT masking a bad join instead of fixing the root cause? |

> **Interview tip:** Narrate your review out loud. Say: "First I'll check the structure... clause order looks correct. Now the logic — I see an INNER JOIN, but the question says 'all customers,' so this should be a LEFT JOIN." This shows your process.

<hr style="border: 3px solid black;">

<a id='5-debugging-workflow'></a>

## 5. Debugging Workflow Under Pressure

When you get a broken query in an interview, follow these 5 steps:

<div class="fc">
  <div class="fc-node fc-start">Step 1: READ the error message<br/>Say it out loud. Identify the category.</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action">Step 2: LOCATE the problem<br/>Go to the LINE number. Look at the caret.</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action">Step 3: ISOLATE the cause<br/>Comment out sections. Run simpler versions.<br/>Does SELECT * FROM table work? Add back one clause.</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action">Step 4: FIX the root cause<br/>Don't patch symptoms. Fix the actual problem.<br/>(DISTINCT is often a patch, not a fix)</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-end">Step 5: VERIFY the fix<br/>Check row count. Check edge cases. Check NULLs.<br/>"Let me verify: I expect N rows because..."</div>
</div>

---

**Isolation techniques:**

| Technique | When to Use |
|---|---|
| Comment out a JOIN | Isolate which join causes fan-out |
| Replace `SELECT *` with `SELECT COUNT(*)` | Quick row count check |
| Add `LIMIT 10` | See sample output before debugging logic |
| `SELECT DISTINCT` test | If DISTINCT fixes the count, you have a join duplication problem |
| Run the subquery alone | Does it return what you expect? |

---

**What to say out loud:**

- "The error says [X], which means [Y]. Let me check line [N]."
- "I'm going to isolate the problem by commenting out the second JOIN."
- "The row count is higher than expected — I think there's a fan-out. Let me check for duplicate keys."
- "I'll verify by checking: I expect [N] rows because [reason]."

> **The #1 rule:** Never silently stare at a query. Talk through your process. Even if you're unsure, narrating your thought process scores points.

<hr style="border: 3px solid black;">

<a id='6-interview-traps'></a>

## 6. Common Interview Traps

These are errors interviewers plant deliberately to see if you catch them:

| Trap | What the Interviewer Planted | What They Want You to Say |
|---|---|---|
| **INNER JOIN when LEFT JOIN needed** | Question says "all customers" but query uses INNER JOIN | "This should be LEFT JOIN since we need all customers, including those with no orders." |
| **COUNT(*) when COUNT(col) needed** | COUNT includes rows that should be excluded (NULLs) | "COUNT(*) counts all rows. If we want only non-NULL values, we should use COUNT(email)." |
| **Missing GROUP BY column** | A column in SELECT isn't in GROUP BY | "employee_name isn't in the GROUP BY — this will error. We need to add it or remove it." |
| **NOT IN with NULLable column** | Subquery could return NULLs, causing zero results | "If manager_id has NULLs, NOT IN will return nothing. I'd use NOT EXISTS instead." |
| **Integer division** | Rate/percentage returns 0 | "This is integer division — 3/5 = 0. I need to cast to DECIMAL first." |
| **Fan-out join** | JOIN duplicates rows, inflating a SUM or COUNT | "This JOIN creates a 1:many relationship. I should aggregate in a CTE first, then join." |
| **WHERE instead of HAVING** | Aggregate function used in WHERE clause | "Aggregate functions can't go in WHERE — they need to be in HAVING, which runs after GROUP BY." |
| **DISTINCT masking bad join** | DISTINCT hides a join duplication problem | "The DISTINCT is suspicious — it might be masking a fan-out. Let me check the join keys." |
| **Missing NULL check in !=** | `WHERE status != 'cancelled'` misses NULL status rows | "This won't include rows where status is NULL. I'd add OR status IS NULL." |

> **Final tip:** In a debugging interview, the process matters as much as the answer. Read the error, classify it, fix the root cause, and verify. Show that you're methodical, not just lucky.